## Change Data Feed (CDF) and Audit Trail

Basel III and FCA regulations require Northbank to maintain a queryable audit trail of all modifications to transaction data. In this exercise, you load payment transactions, simulate data corrections (a common real-world event), and then query the Change Data Feed to produce a full audit log.

### Task 3.1 - Load Initial Transactions

Run the cell below to insert 10 payment transactions into `fact_transactions`. These represent one day's batch of processed payments.

In [0]:
%sql
INSERT INTO banking_lab.fact_transactions VALUES
  ('TXN-001', 'ACC-001', 'C-1001', 'Credit',  1500.00, 'GBP', '2025-11-01', 'Salary payment'),
  ('TXN-002', 'ACC-001', 'C-1001', 'Debit',    120.50, 'GBP', '2025-11-03', 'Amazon purchase'),
  ('TXN-003', 'ACC-002', 'C-1002', 'Debit',    250.00, 'GBP', '2025-11-05', 'Grocery store'),
  ('TXN-004', 'ACC-003', 'C-1003', 'Credit',  3200.00, 'GBP', '2025-11-07', 'Salary payment'),
  ('TXN-005', 'ACC-003', 'C-1003', 'Transfer', 500.00, 'GBP', '2025-11-10', 'Rent transfer'),
  ('TXN-006', 'ACC-004', 'C-1004', 'Debit',     89.99, 'GBP', '2025-11-12', 'Utility bill'),
  ('TXN-007', 'ACC-005', 'C-1005', 'Debit',   1500.00, 'GBP', '2025-11-14', 'Business expense'),
  ('TXN-008', 'ACC-006', 'C-1006', 'Credit',  2800.00, 'GBP', '2025-11-15', 'Salary payment'),
  ('TXN-009', 'ACC-007', 'C-1007', 'Debit',    340.00, 'GBP', '2025-11-18', 'Car insurance'),
  ('TXN-010', 'ACC-008', 'C-1008', 'Transfer', 200.00, 'GBP', '2025-11-20', 'Transfer to savings');

### Task 3.2 - Simulate transaction corrections

Northbank's reconciliation team identified two amount discrepancies. Run the cell below to apply the corrections.

In [0]:
%sql
-- Correction 1: TXN-003 amount was mis-keyed (correct: £275.50, not £250.00)
UPDATE banking_lab.fact_transactions
SET amount = 275.50, description = 'Grocery store (corrected)'
WHERE transaction_id = 'TXN-003';

In [0]:
%sql
-- Correction 2: TXN-007 amount is under dispute
UPDATE banking_lab.fact_transactions
SET amount = 1450.00, description = 'Business expense (disputed amount)'
WHERE transaction_id = 'TXN-007';

### Task 3.3 - Query the Change Data Feed to build an Audit Trail

Change Data Feed tracks **every change made to the table**.

Using `table_changes()`:
- We retrieve inserts, updates, and deletes
- We also get metadata:
  - `_change_type`
  - `_commit_version`
  - `_commit_timestamp`

This creates a **complete audit log of all changes**.

Perfect for:
- Regulatory compliance
- Debugging data pipelines

In [0]:
%sql
SELECT
    transaction_id,
    amount,
    description,
    _change_type,
    _commit_version,
    _commit_timestamp
FROM table_changes('banking_lab.fact_transactions', 0)
ORDER BY _commit_timestamp, transaction_id;

### Task 3.4 - Isolate Corrected Records Only

Each UPDATE generates:
- `update_preimage` → before change
- `update_postimage` → after change

We filter for:
👉 `_change_type = 'update_postimage'`

This gives us the **final corrected state of records**.

Used for:
- Compliance reporting (FCA)
- Audit summaries

In [0]:
%sql
SELECT
    transaction_id,
    amount,
    description,
    _commit_timestamp
FROM table_changes('banking_lab.fact_transactions', 0)
WHERE _change_type = 'update_postimage';

### Exercise 4 - Delta Lake Time Travel

Delta Lake maintains a full transaction log of every change made to a table. This gives you the ability to query previous versions of your data and even restore a table to an earlier state — without restoring from backup. This is critical for Northbank when transaction data is accidentally modified.


### Task 4.1 - Inspect the Transaction History

Delta Lake maintains a full transaction log.

Using `DESCRIBE HISTORY`, we can see:
- All operations (INSERT, UPDATE, DELETE)
- Version numbers
- Timestamps

Each version represents a **snapshot of the table**.

In [0]:
%sql
DESCRIBE HISTORY banking_lab.fact_transactions;

### Task 4.2 - Query original transaction amounts before corrections

Using `VERSION AS OF`, we can query past data.

This allows us to:
- See data before updates
- Compare historical vs current state

No backups required — Delta handles this automatically.

In [0]:
%sql
SELECT transaction_id, amount, description
FROM banking_lab.fact_transactions VERSION AS OF 1
WHERE transaction_id IN ('TXN-003', 'TXN-007');

### Task 4.3 - Restore the table to its pre-correction state

We can roll back the entire table to a previous version.

This is useful when:
- Incorrect updates were applied
- Data corruption occurs

Delta Lake enables:
👉 Instant rollback without restoring backups

In [0]:
%sql
RESTORE TABLE banking_lab.fact_transactions
TO VERSION AS OF 1;

### Task 4.4 - Verify Restore

In [0]:
%sql
SELECT transaction_id, amount, description
FROM banking_lab.fact_transactions
WHERE transaction_id IN ('TXN-003', 'TXN-007');